Below is a complete, end-to-end NeMo → ONNX → TensorRT pipeline that gives you:

FP16 (half precision) inference (PyTorch + TensorRT)

INT8 inference (post-training quantization with calibration → TensorRT engine)

This is written for NeMo ASR CTC models (Conformer-CTC / Parakeet-CTC). NeMo’s CTC models are typically loaded via from_pretrained() (examples on NVIDIA HF pages) . NeMo also provides an export pathway via scripts/export.py .

For TensorRT, note:

TensorRT supports FP16 and INT8 (among others)

In INT8 mode, you should use a calibration cache (otherwise dynamic ranges can be wrong)

The preferred calibrator class is IInt8EntropyCalibrator2

What you’ll run (high level)

Load a NeMo CTC ASR model (from HF/NGC name or .nemo file)

Compute mel features using the model’s own preprocessor

Export encoder+decoder to ONNX (features → logits)

Build TensorRT engine:

FP16 engine

INT8 engine with calibration (audio folder → feature batches → calib cache)

Run inference with:

PyTorch FP16

TensorRT FP16 / INT8

Environment notes (quick)

Easiest is using an NVIDIA container (NeMo docs recommend container runtime installs)

trtexec is typically available in TensorRT/NGC containers at /opt/tensorrt/bin/trtexec
(You don’t need trtexec here; we build engines in Python, but it’s useful for debugging.)

In [ ]:
#!/usr/bin/env python3
"""
NeMo CTC ASR -> ONNX -> TensorRT FP16/INT8 engines + inference

Supports:
- Loading NeMo CTC model from:
    * HF/NGC name (e.g. "nvidia/stt_en_conformer_ctc_large")
    * local .nemo checkpoint path
- ONNX export of encoder+decoder (features -> logits)
- TensorRT engine build:
    * FP16
    * INT8 (PTQ calibration using IInt8EntropyCalibrator2 + cache)
- Inference:
    * PyTorch FP16
    * TensorRT FP16 / INT8

Notes:
- Designed for NeMo CTC models (EncDecCTCModel / EncDecCTCModelBPE).
- For RNNT/TDT models, export/build differs (encoder + predictor + joint).
"""

import os
import sys
import glob
import json
import math
import time
import argparse
import inspect
from dataclasses import dataclass
from typing import Iterator, List, Optional, Tuple

import numpy as np

import torch
import torch.nn as nn

# Audio IO
import soundfile as sf

# Optional resample
try:
    import torchaudio
except Exception:
    torchaudio = None

# TensorRT + CUDA
try:
    import tensorrt as trt
    import pycuda.driver as cuda
    import pycuda.autoinit  # noqa: F401 (initializes CUDA context)
except Exception:
    trt = None
    cuda = None


# ---------------------------
# NeMo loader
# ---------------------------
def load_nemo_ctc_model(model_name_or_path: str, device: str = "cuda"):
    import nemo.collections.asr as nemo_asr

    map_location = torch.device(device)

    if os.path.exists(model_name_or_path) and model_name_or_path.endswith(".nemo"):
        model = nemo_asr.models.ASRModel.restore_from(model_name_or_path, map_location=map_location)
    else:
        # Try CTC-BPE first (Parakeet-CTC, many Conformer-CTC BPE)
        try:
            model = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(
                model_name=model_name_or_path,
                map_location=map_location,
            )
        except Exception:
            # Fallback to generic ASRModel loader
            model = nemo_asr.models.ASRModel.from_pretrained(
                model_name=model_name_or_path,
                map_location=map_location,
            )

    model.eval()
    model.to(map_location)
    return model


# ---------------------------
# Audio + features
# ---------------------------
def load_audio_mono(path: str) -> Tuple[np.ndarray, int]:
    wav, sr = sf.read(path, dtype="float32", always_2d=False)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)  # mono
    return wav, sr


def resample_if_needed(wav: np.ndarray, sr: int, target_sr: int) -> Tuple[np.ndarray, int]:
    if sr == target_sr:
        return wav, sr
    if torchaudio is None:
        raise RuntimeError("torchaudio not available for resampling. Install torchaudio or provide target_sr audio.")
    x = torch.from_numpy(wav).unsqueeze(0)  # [1, T]
    y = torchaudio.functional.resample(x, orig_freq=sr, new_freq=target_sr)
    return y.squeeze(0).cpu().numpy(), target_sr


@torch.no_grad()
def nemo_preprocess_to_features(asr_model, wav: np.ndarray, sr: int, device: str = "cuda"):
    # Determine model sample rate if possible
    target_sr = None
    try:
        target_sr = int(asr_model.preprocessor.featurizer.sample_rate)
    except Exception:
        # common default
        target_sr = 16000

    wav, _ = resample_if_needed(wav, sr, target_sr)

    # NeMo preprocessor expects: input_signal [B, T], length [B]
    x = torch.from_numpy(wav).to(device).unsqueeze(0)  # [1, T]
    x_len = torch.tensor([x.shape[1]], device=device, dtype=torch.int64)

    # Many NeMo preprocessors use keyword args input_signal/length
    feats, feat_lens = asr_model.preprocessor(input_signal=x, length=x_len)

    # feats typically [B, n_mels, Tfeat], feat_lens [B]
    return feats, feat_lens


# ---------------------------
# Export wrapper: (features -> logits)
# ---------------------------
def _try_call(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except TypeError:
        return None


def call_encoder(encoder, feats, feat_lens):
    # Try several common signatures across NeMo versions/modules
    out = _try_call(encoder, audio_signal=feats, length=feat_lens)
    if out is not None:
        return out
    out = _try_call(encoder, feats, feat_lens)
    if out is not None:
        return out
    out = _try_call(encoder, input_signal=feats, input_signal_length=feat_lens)
    if out is not None:
        return out
    raise TypeError("Could not call encoder with known signatures.")


def call_decoder(decoder, enc, enc_lens):
    # Decoder often ignores lengths; try with/without lengths.
    out = _try_call(decoder, enc, enc_lens)
    if out is not None:
        return out
    out = _try_call(decoder, enc)
    if out is not None:
        return out
    raise TypeError("Could not call decoder with known signatures.")


class CTCNetFeaturesToLogits(nn.Module):
    """
    Exports encoder+decoder. Input is mel features (not raw audio).
    Output is logits (NOT log_softmax), plus encoded lengths.

    This avoids ambiguity across NeMo versions about whether decoder returns
    logits vs log_probs.
    """

    def __init__(self, asr_model):
        super().__init__()
        self.encoder = asr_model.encoder
        self.decoder = asr_model.decoder

    def forward(self, feats: torch.Tensor, feat_lens: torch.Tensor):
        enc, enc_lens = call_encoder(self.encoder, feats, feat_lens)

        dec_out = call_decoder(self.decoder, enc, enc_lens)

        # Some decoders return tuple/list
        if isinstance(dec_out, (tuple, list)):
            logits = dec_out[0]
        else:
            logits = dec_out

        return logits, enc_lens


# ---------------------------
# ONNX export
# ---------------------------
@torch.no_grad()
def export_onnx_ctc(
    asr_model,
    onnx_path: str,
    example_feats: torch.Tensor,
    example_lens: torch.Tensor,
    opset: int = 17,
):
    wrapper = CTCNetFeaturesToLogits(asr_model).eval()

    # Force CPU export graph? Better keep on CUDA if supported. We'll keep on same device.
    device = example_feats.device
    wrapper.to(device)

    # Use int32 lengths for TRT friendliness
    example_lens_i32 = example_lens.to(dtype=torch.int32)

    input_names = ["feats", "feat_lens"]
    output_names = ["logits", "enc_lens"]

    # Dynamic axes: time dimension dynamic; batch dynamic (optional)
    dynamic_axes = {
        "feats": {0: "B", 2: "T"},
        "feat_lens": {0: "B"},
        "logits": {0: "B", 1: "T_out"},
        "enc_lens": {0: "B"},
    }

    torch.onnx.export(
        wrapper,
        (example_feats, example_lens_i32),
        onnx_path,
        export_params=True,
        opset_version=opset,
        do_constant_folding=True,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
    )
    print(f"[OK] Exported ONNX -> {onnx_path}")


# ---------------------------
# TensorRT helpers
# ---------------------------
def trt_logger():
    return trt.Logger(trt.Logger.INFO)


def set_workspace(config, workspace_mb: int):
    # TRT 8.x: config.max_workspace_size
    # TRT 9/10: set_memory_pool_limit preferred
    bytes_ = int(workspace_mb) * 1024 * 1024
    if hasattr(config, "set_memory_pool_limit") and hasattr(trt, "MemoryPoolType"):
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, bytes_)
    else:
        config.max_workspace_size = bytes_


def build_trt_engine_from_onnx(
    onnx_path: str,
    engine_path: str,
    fp16: bool,
    int8: bool,
    calibrator=None,
    min_T: int = 64,
    opt_T: int = 512,
    max_T: int = 2048,
    n_mels: int = 80,
    batch_size: int = 1,
    workspace_mb: int = 4096,
):
    if trt is None:
        raise RuntimeError("TensorRT not available. Install TensorRT + pycuda in a CUDA environment.")

    logger = trt_logger()
    builder = trt.Builder(logger)

    flags = 1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)
    network = builder.create_network(flags)
    parser = trt.OnnxParser(network, logger)

    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            print("[ERR] Failed to parse ONNX. Errors:")
            for i in range(parser.num_errors):
                print(parser.get_error(i))
            raise RuntimeError("ONNX parse failed")

    config = builder.create_builder_config()
    set_workspace(config, workspace_mb)

    if fp16:
        config.set_flag(trt.BuilderFlag.FP16)

    if int8:
        config.set_flag(trt.BuilderFlag.INT8)
        if calibrator is None:
            raise ValueError("INT8 requested but no calibrator provided")
        config.int8_calibrator = calibrator  # uses IInt8EntropyCalibrator2, etc.

    # Optimization profile for dynamic shapes
    profile = builder.create_optimization_profile()

    # Identify input indices/names
    n_inputs = network.num_inputs
    input_names = [network.get_input(i).name for i in range(n_inputs)]

    # Expect inputs: feats [B, n_mels, T], feat_lens [B]
    # (Names come from export: "feats", "feat_lens")
    feats_name = "feats" if "feats" in input_names else input_names[0]
    lens_name = "feat_lens" if "feat_lens" in input_names else input_names[1]

    profile.set_shape(
        feats_name,
        min=(batch_size, n_mels, min_T),
        opt=(batch_size, n_mels, opt_T),
        max=(batch_size, n_mels, max_T),
    )
    profile.set_shape(
        lens_name,
        min=(batch_size,),
        opt=(batch_size,),
        max=(batch_size,),
    )

    config.add_optimization_profile(profile)

    # Build engine
    t0 = time.time()
    engine = builder.build_engine(network, config)
    dt = time.time() - t0
    if engine is None:
        raise RuntimeError("Failed to build TensorRT engine")

    with open(engine_path, "wb") as f:
        f.write(engine.serialize())

    print(f"[OK] Built engine ({'FP16' if fp16 else 'FP32'}{' + INT8' if int8 else ''}) -> {engine_path}  ({dt:.1f}s)")
    return engine_path


# ---------------------------
# INT8 Calibrator
# ---------------------------
class AudioFeatureBatcher:
    """
    Generates fixed-shape feature batches from a folder of wav files.
    We pad/truncate features to opt_T so calibration batches are consistent.
    """

    def __init__(
        self,
        asr_model,
        wav_dir: str,
        device: str,
        batch_size: int,
        n_mels: int,
        opt_T: int,
        max_files: Optional[int] = None,
    ):
        self.asr_model = asr_model
        self.device = device
        self.batch_size = batch_size
        self.n_mels = n_mels
        self.opt_T = opt_T

        wavs = sorted(glob.glob(os.path.join(wav_dir, "**/*.wav"), recursive=True))
        if max_files is not None:
            wavs = wavs[:max_files]
        if not wavs:
            raise ValueError(f"No .wav files found under {wav_dir}")
        self.wavs = wavs
        self.idx = 0

    def __iter__(self) -> Iterator[Tuple[np.ndarray, np.ndarray]]:
        return self

    def __next__(self) -> Tuple[np.ndarray, np.ndarray]:
        if self.idx >= len(self.wavs):
            raise StopIteration

        feats_batch = []
        lens_batch = []

        for _ in range(self.batch_size):
            if self.idx >= len(self.wavs):
                break
            p = self.wavs[self.idx]
            self.idx += 1

            wav, sr = load_audio_mono(p)
            feats, feat_lens = nemo_preprocess_to_features(self.asr_model, wav, sr, device=self.device)
            feats = feats.squeeze(0)         # [n_mels, T]
            feat_lens = int(feat_lens.item())

            # Pad/truncate to opt_T
            T = feats.shape[-1]
            if T >= self.opt_T:
                feats = feats[:, : self.opt_T]
                feat_lens = min(feat_lens, self.opt_T)
            else:
                pad = self.opt_T - T
                feats = torch.nn.functional.pad(feats, (0, pad), mode="constant", value=0.0)

            # Ensure [n_mels, opt_T]
            feats = feats[: self.n_mels, : self.opt_T]

            feats_batch.append(feats.cpu().numpy().astype(np.float32))
            lens_batch.append(np.int32(feat_lens))

        if not feats_batch:
            raise StopIteration

        feats_np = np.stack(feats_batch, axis=0)     # [B, n_mels, opt_T]
        lens_np = np.array(lens_batch, dtype=np.int32)  # [B]
        return feats_np, lens_np


class EntropyCalibrator2(trt.IInt8EntropyCalibrator2):
    """
    TensorRT INT8 calibrator.
    Docs: IInt8EntropyCalibrator2 (preferred calibrator) :contentReference[oaicite:7]{index=7}
    """

    def __init__(self, batcher: AudioFeatureBatcher, cache_file: str):
        super().__init__()
        self.batcher = iter(batcher)
        self.cache_file = cache_file

        # Allocate device buffers once (based on opt shapes)
        # We will infer shapes from the first batch.
        first_feats, first_lens = next(self.batcher)
        self._feats_shape = first_feats.shape
        self._lens_shape = first_lens.shape

        self.device_input_feats = cuda.mem_alloc(first_feats.nbytes)
        self.device_input_lens = cuda.mem_alloc(first_lens.nbytes)

        # Reset batcher with the first batch pre-stored
        self._first_batch = (first_feats, first_lens)
        self._used_first = False

    def get_batch_size(self):
        return self._feats_shape[0]

    def get_batch(self, names):
        try:
            if not self._used_first:
                feats, lens = self._first_batch
                self._used_first = True
            else:
                feats, lens = next(self.batcher)
        except StopIteration:
            return None

        cuda.memcpy_htod(self.device_input_feats, feats)
        cuda.memcpy_htod(self.device_input_lens, lens)

        # Return pointers in the same order as TensorRT binds inputs.
        # Names are the network input names.
        ptrs = []
        for n in names:
            if "feats" in n:
                ptrs.append(int(self.device_input_feats))
            else:
                ptrs.append(int(self.device_input_lens))
        return ptrs

    def read_calibration_cache(self):
        if os.path.exists(self.cache_file):
            with open(self.cache_file, "rb") as f:
                return f.read()
        return None

    def write_calibration_cache(self, cache):
        with open(self.cache_file, "wb") as f:
            f.write(cache)


# ---------------------------
# TensorRT inference
# ---------------------------
@dataclass
class TrtBindings:
    inputs: dict
    outputs: dict
    bindings: list
    stream: any


def allocate_trt_io(engine, context, feats_shape, lens_shape):
    """
    Allocate device buffers for a given (dynamic) input shape.
    """
    bindings = [None] * engine.num_bindings
    inputs = {}
    outputs = {}
    stream = cuda.Stream()

    for i in range(engine.num_bindings):
        name = engine.get_binding_name(i)
        dtype = trt.nptype(engine.get_binding_dtype(i))
        is_input = engine.binding_is_input(i)

        if is_input:
            if name == "feats":
                context.set_binding_shape(i, feats_shape)
                host = cuda.pagelocked_empty(trt.volume(feats_shape), dtype)
            else:
                context.set_binding_shape(i, lens_shape)
                host = cuda.pagelocked_empty(trt.volume(lens_shape), dtype)
        else:
            out_shape = tuple(context.get_binding_shape(i))
            host = cuda.pagelocked_empty(trt.volume(out_shape), dtype)

        device = cuda.mem_alloc(host.nbytes)
        bindings[i] = int(device)

        if is_input:
            inputs[name] = (host, device)
        else:
            outputs[name] = (host, device)

    return TrtBindings(inputs=inputs, outputs=outputs, bindings=bindings, stream=stream)


def trt_infer(engine_path: str, feats_np: np.ndarray, lens_np: np.ndarray):
    logger = trt_logger()
    with open(engine_path, "rb") as f, trt.Runtime(logger) as runtime:
        engine = runtime.deserialize_cuda_engine(f.read())
        if engine is None:
            raise RuntimeError("Failed to deserialize engine")

        context = engine.create_execution_context()
        if context is None:
            raise RuntimeError("Failed to create execution context")

        feats_shape = feats_np.shape  # [B, n_mels, T]
        lens_shape = lens_np.shape    # [B]

        io = allocate_trt_io(engine, context, feats_shape, lens_shape)

        # Copy inputs
        feats_host, feats_dev = io.inputs["feats"]
        lens_host, lens_dev = io.inputs["feat_lens"]

        np.copyto(feats_host, feats_np.ravel())
        np.copyto(lens_host, lens_np.ravel())

        cuda.memcpy_htod_async(feats_dev, feats_host, io.stream)
        cuda.memcpy_htod_async(lens_dev, lens_host, io.stream)

        # Execute
        context.execute_async_v2(bindings=io.bindings, stream_handle=io.stream.handle)

        # Copy outputs back
        out_dict = {}
        for name, (host, dev) in io.outputs.items():
            cuda.memcpy_dtoh_async(host, dev, io.stream)

        io.stream.synchronize()

        for name, (host, dev) in io.outputs.items():
            out_dict[name] = host.copy()

        # Reshape outputs
        # Output names from export: "logits", "enc_lens"
        logits = out_dict.get("logits", None)
        enc_lens = out_dict.get("enc_lens", None)

        if logits is None or enc_lens is None:
            # fallback: pick first 2 outputs
            outs = list(out_dict.items())
            logits = outs[0][1]
            enc_lens = outs[1][1]

        # Determine runtime output shapes via context
        # Find binding index by name
        def binding_shape_by_name(n):
            for i in range(engine.num_bindings):
                if engine.get_binding_name(i) == n:
                    return tuple(context.get_binding_shape(i))
            return None

        logits_shape = binding_shape_by_name("logits")
        enc_lens_shape = binding_shape_by_name("enc_lens")

        if logits_shape is None:
            # best guess: [B, T_out, V]
            B = feats_np.shape[0]
            # can't infer T_out,V safely without binding shapes. Keep flat.
            logits_np = logits
        else:
            logits_np = logits.reshape(logits_shape)

        if enc_lens_shape is None:
            enc_lens_np = enc_lens
        else:
            enc_lens_np = enc_lens.reshape(enc_lens_shape)

        return logits_np, enc_lens_np


# ---------------------------
# CTC greedy decoding
# ---------------------------
def get_blank_id(asr_model) -> int:
    # Many NeMo decoders store blank index here
    if hasattr(asr_model, "decoder") and hasattr(asr_model.decoder, "blank_idx"):
        return int(asr_model.decoder.blank_idx)
    # fallback
    return 0


def ctc_greedy_decode(asr_model, logits: np.ndarray, enc_lens: np.ndarray) -> str:
    """
    logits: [B, T, V] (or [B, V, T] occasionally)
    """
    # Ensure batch=1
    if logits.ndim != 3:
        raise ValueError(f"Expected 3D logits, got shape {logits.shape}")

    B = logits.shape[0]
    assert B == 1, "This helper decodes batch_size=1 for simplicity."

    # Fix layout if needed
    # If second dim looks like vocab and third like time, transpose
    V_guess = None
    if hasattr(asr_model, "decoder") and hasattr(asr_model.decoder, "num_classes"):
        V_guess = int(asr_model.decoder.num_classes)
    if V_guess is not None and logits.shape[1] == V_guess:
        logits = np.transpose(logits, (0, 2, 1))  # [B, T, V]

    log_probs = torch.log_softmax(torch.from_numpy(logits), dim=-1)  # [1, T, V]
    pred = torch.argmax(log_probs, dim=-1).squeeze(0).cpu().tolist()

    T_out = int(enc_lens[0]) if np.ndim(enc_lens) > 0 else len(pred)
    pred = pred[:T_out]

    blank = get_blank_id(asr_model)

    # CTC collapse
    collapsed = []
    prev = None
    for p in pred:
        if p == blank:
            prev = p
            continue
        if prev is None or p != prev:
            collapsed.append(p)
        prev = p

    # Tokenizer-based (BPE) vs char-based
    if hasattr(asr_model, "tokenizer") and asr_model.tokenizer is not None:
        # Many NeMo BPE tokenizers support ids_to_text
        try:
            return asr_model.tokenizer.ids_to_text(collapsed)
        except Exception:
            pass

    # Fallback: if model has decoding module, try it
    try:
        # Some NeMo models expose a decoding helper
        hyp = asr_model.decoding.ctc_decoder_predictions_tensor(
            log_probs=log_probs,
            encoded_lengths=torch.tensor([T_out]),
        )
        # hyp is list[str] or list[Hypothesis]
        if isinstance(hyp, list) and hyp:
            if isinstance(hyp[0], str):
                return hyp[0]
            if hasattr(hyp[0], "text"):
                return hyp[0].text
    except Exception:
        pass

    # Worst-case fallback: just return ids
    return " ".join(map(str, collapsed))


# ---------------------------
# CLI
# ---------------------------
def cmd_export(args):
    device = args.device
    asr = load_nemo_ctc_model(args.model, device=device)

    wav, sr = load_audio_mono(args.example_wav)
    feats, feat_lens = nemo_preprocess_to_features(asr, wav, sr, device=device)
    print(f"[INFO] Example feats shape: {tuple(feats.shape)}  feat_lens: {feat_lens.tolist()}")

    os.makedirs(os.path.dirname(args.onnx) or ".", exist_ok=True)
    export_onnx_ctc(asr, args.onnx, feats, feat_lens, opset=args.opset)


def cmd_build_fp16(args):
    device = args.device
    asr = load_nemo_ctc_model(args.model, device=device)

    # Infer n_mels from preprocessor output (using example wav)
    wav, sr = load_audio_mono(args.example_wav)
    feats, feat_lens = nemo_preprocess_to_features(asr, wav, sr, device=device)
    n_mels = int(feats.shape[1])

    build_trt_engine_from_onnx(
        onnx_path=args.onnx,
        engine_path=args.engine,
        fp16=True,
        int8=False,
        calibrator=None,
        min_T=args.min_T,
        opt_T=args.opt_T,
        max_T=args.max_T,
        n_mels=n_mels,
        batch_size=args.batch_size,
        workspace_mb=args.workspace_mb,
    )


def cmd_build_int8(args):
    device = args.device
    asr = load_nemo_ctc_model(args.model, device=device)

    # Infer n_mels from preprocessor output (example wav)
    wav, sr = load_audio_mono(args.example_wav)
    feats, feat_lens = nemo_preprocess_to_features(asr, wav, sr, device=device)
    n_mels = int(feats.shape[1])

    os.makedirs(os.path.dirname(args.calib_cache) or ".", exist_ok=True)

    batcher = AudioFeatureBatcher(
        asr_model=asr,
        wav_dir=args.calib_wav_dir,
        device=device,
        batch_size=args.batch_size,
        n_mels=n_mels,
        opt_T=args.opt_T,
        max_files=args.max_calib_files,
    )
    calib = EntropyCalibrator2(batcher=batcher, cache_file=args.calib_cache)

    build_trt_engine_from_onnx(
        onnx_path=args.onnx,
        engine_path=args.engine,
        fp16=args.fp16,   # often keep FP16 enabled as well
        int8=True,
        calibrator=calib,
        min_T=args.min_T,
        opt_T=args.opt_T,
        max_T=args.max_T,
        n_mels=n_mels,
        batch_size=args.batch_size,
        workspace_mb=args.workspace_mb,
    )


def cmd_infer_pytorch_fp16(args):
    device = args.device
    asr = load_nemo_ctc_model(args.model, device=device)

    wav, sr = load_audio_mono(args.wav)
    feats, feat_lens = nemo_preprocess_to_features(asr, wav, sr, device=device)

    net = CTCNetFeaturesToLogits(asr).to(device).eval().half()
    feats16 = feats.half()
    lens32 = feat_lens.to(dtype=torch.int32)

    with torch.inference_mode():
        logits, enc_lens = net(feats16, lens32)

    text = ctc_greedy_decode(asr, logits.float().cpu().numpy(), enc_lens.cpu().numpy())
    print(text)


def cmd_infer_trt(args):
    device = args.device
    asr = load_nemo_ctc_model(args.model, device=device)

    wav, sr = load_audio_mono(args.wav)
    feats, feat_lens = nemo_preprocess_to_features(asr, wav, sr, device=device)

    # Pad/truncate feats to <= max_T (engine profile max) to avoid shape errors
    B, n_mels, T = feats.shape
    T_max = args.max_T
    if T > T_max:
        feats = feats[:, :, :T_max]
        feat_lens = torch.clamp(feat_lens, max=T_max)
    elif T < args.min_T:
        # pad up to min_T
        pad = args.min_T - T
        feats = torch.nn.functional.pad(feats, (0, pad), mode="constant", value=0.0)
        T = feats.shape[-1]

    feats_np = feats.squeeze(0).cpu().numpy().astype(np.float32)  # [n_mels, T]
    feats_np = np.expand_dims(feats_np, axis=0)                   # [1, n_mels, T]
    lens_np = np.array([int(feat_lens.item())], dtype=np.int32)   # [1]

    logits_np, enc_lens_np = trt_infer(args.engine, feats_np, lens_np)
    text = ctc_greedy_decode(asr, logits_np, enc_lens_np)
    print(text)


def build_argparser():
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest="cmd", required=True)

    # export
    s = sub.add_parser("export_onnx")
    s.add_argument("--model", required=True, help="HF/NGC model name or local .nemo path")
    s.add_argument("--example_wav", required=True, help="A .wav file to infer feature shapes for export")
    s.add_argument("--onnx", required=True)
    s.add_argument("--opset", type=int, default=17)
    s.add_argument("--device", default="cuda")
    s.set_defaults(func=cmd_export)

    # build fp16
    s = sub.add_parser("build_fp16")
    s.add_argument("--model", required=True)
    s.add_argument("--example_wav", required=True)
    s.add_argument("--onnx", required=True)
    s.add_argument("--engine", required=True)
    s.add_argument("--batch_size", type=int, default=1)
    s.add_argument("--min_T", type=int, default=64)
    s.add_argument("--opt_T", type=int, default=512)
    s.add_argument("--max_T", type=int, default=2048)
    s.add_argument("--workspace_mb", type=int, default=4096)
    s.add_argument("--device", default="cuda")
    s.set_defaults(func=cmd_build_fp16)

    # build int8
    s = sub.add_parser("build_int8")
    s.add_argument("--model", required=True)
    s.add_argument("--example_wav", required=True)
    s.add_argument("--onnx", required=True)
    s.add_argument("--engine", required=True)
    s.add_argument("--calib_wav_dir", required=True, help="Folder of .wav files for INT8 calibration")
    s.add_argument("--calib_cache", required=True, help="Where to write/read calibration cache")
    s.add_argument("--max_calib_files", type=int, default=500)
    s.add_argument("--batch_size", type=int, default=1)
    s.add_argument("--min_T", type=int, default=64)
    s.add_argument("--opt_T", type=int, default=512)
    s.add_argument("--max_T", type=int, default=2048)
    s.add_argument("--workspace_mb", type=int, default=4096)
    s.add_argument("--fp16", action="store_true", help="Also enable FP16 in INT8 build (often good)")
    s.add_argument("--device", default="cuda")
    s.set_defaults(func=cmd_build_int8)

    # infer pytorch fp16
    s = sub.add_parser("infer_pytorch_fp16")
    s.add_argument("--model", required=True)
    s.add_argument("--wav", required=True)
    s.add_argument("--device", default="cuda")
    s.set_defaults(func=cmd_infer_pytorch_fp16)

    # infer trt
    s = sub.add_parser("infer_trt")
    s.add_argument("--model", required=True)
    s.add_argument("--engine", required=True)
    s.add_argument("--wav", required=True)
    s.add_argument("--min_T", type=int, default=64)
    s.add_argument("--max_T", type=int, default=2048)
    s.add_argument("--device", default="cuda")
    s.set_defaults(func=cmd_infer_trt)

    return p


def main():
    args = build_argparser().parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


How to run (example with Conformer-CTC)

Pick a model. For example, NVIDIA provides Conformer-CTC checkpoints on HF like nvidia/stt_en_conformer_ctc_large

1) Export ONNX

In [ ]:
python nemo_ctc_trt_quant.py export_onnx \
  --model nvidia/stt_en_conformer_ctc_large \
  --example_wav samples.wav \
  --onnx conformer_ctc_feats2logits.onnx


2) Build FP16 TensorRT engine

In [ ]:
python nemo_ctc_trt_quant.py build_fp16 \
  --model nvidia/stt_en_conformer_ctc_large \
  --example_wav samples.wav \
  --onnx conformer_ctc_feats2logits.onnx \
  --engine conformer_ctc_fp16.engine \
  --min_T 64 --opt_T 512 --max_T 2048 \
  --batch_size 1


3) Build INT8 TensorRT engine (PTQ + calibration)

You need a folder of representative .wav files.

Why cache matters: TensorRT’s CLI docs explicitly warn that in INT8 mode you should provide calibration info (otherwise ranges can be random / accuracy off)

In [ ]:
python nemo_ctc_trt_quant.py build_int8 \
  --model nvidia/stt_en_conformer_ctc_large \
  --example_wav samples.wav \
  --onnx conformer_ctc_feats2logits.onnx \
  --engine conformer_ctc_int8.engine \
  --calib_wav_dir calib_wavs/ \
  --calib_cache conformer_ctc_int8.cache \
  --max_calib_files 500 \
  --min_T 64 --opt_T 512 --max_T 2048 \
  --batch_size 1 \
  --fp16


4) Inference
PyTorch FP16 inference

In [ ]:
python nemo_ctc_trt_quant.py infer_pytorch_fp16 \
  --model nvidia/stt_en_conformer_ctc_large \
  --wav test.wav


TensorRT inference (FP16 or INT8 engine)

(or use conformer_ctc_int8.engine)

In [ ]:
python nemo_ctc_trt_quant.py infer_trt \
  --model nvidia/stt_en_conformer_ctc_large \
  --engine conformer_ctc_fp16.engine \
  --wav test.wav \
  --min_T 64 --max_T 2048


Practical tips (so this works cleanly)

Choose T-profile well: max_T must cover your longest audio after feature extraction. If you plan 30s audio, your feature T can be a few thousand; adjust max_T.

Calibration set quality matters: INT8 accuracy depends on calibration distribution; use real target-domain audio.

CTC models only here: If you want RNNT/TDT, export differs (encoder + predictor + joint), and the engine build needs 2–3 subgraphs. NeMo’s export tooling is documented via scripts/export.py